# Propensity Analysis – Multi‑Model Feature Accretion
This notebook loads *all* JSONL outputs and compares models on novelty, repetition, and breakdown signals.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')

OUTPUT_DIR = Path('experiments/outputs')
files = sorted(OUTPUT_DIR.glob('local_llm__*.jsonl'))

records = []
summaries = []
for path in files:
    model = path.stem.replace('local_llm__', '')
    for line in path.read_text().splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)
        rec['model'] = model
        if rec.get('record_type') == 'prompt_response':
            records.append(rec)
        elif rec.get('record_type') == 'run_summary':
            summaries.append(rec)

df = pd.DataFrame(records)
df_summary = pd.DataFrame(summaries)
# Focus on feature_accretion runs if present
if 'mode' in df.columns:
    df = df[df['mode'] == 'feature_accretion']
df.head()


In [ ]:
df_summary

## Quick Metrics by Model

In [ ]:
metrics = df.groupby('model').agg(
    steps=('step','max'),
    avg_novelty=('novelty_score','mean'),
    avg_repetition=('repetition_rate','mean'),
    avg_similarity=('similarity_to_prev','mean'),
    hit_max_rate=('hit_max_gen_tokens','mean'),
    format_ok_rate=('format_ok','mean') if 'format_ok' in df else ('step','count')
)
metrics


## Novelty Over Time (per model)

In [ ]:
plt.figure(figsize=(10,5))
sns.lineplot(data=df, x='step', y='novelty_score', hue='model')
plt.axhline(0.25, color='red', linestyle='--', label='novelty threshold')
plt.title('Novelty score by step')
plt.legend()
plt.show()

## Repetition and Similarity Trends

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
sns.lineplot(data=df, x='step', y='repetition_rate', hue='model', ax=ax[0])
ax[0].set_title('Repetition rate')
sns.lineplot(data=df, x='step', y='similarity_to_prev', hue='model', ax=ax[1])
ax[1].set_title('Similarity to previous')
plt.tight_layout()
plt.show()

## Breakdown Score and Truncation

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
sns.lineplot(data=df, x='step', y='breakdown_score', hue='model', ax=ax[0])
ax[0].set_title('Breakdown score')
sns.lineplot(data=df, x='step', y='hit_max_gen_tokens', hue='model', ax=ax[1])
ax[1].set_title('Hit max gen tokens (rate)')
plt.tight_layout()
plt.show()

## Feature Ledger Growth

In [ ]:
if 'feature_ledger_count' in df:
    plt.figure(figsize=(10,4))
    sns.lineplot(data=df, x='step', y='feature_ledger_count', hue='model', marker='o')
    plt.title('Feature ledger count')
    plt.show()

## Memory Check – Language Recall

In [ ]:
if 'memory_match' in df:
    plt.figure(figsize=(10,4))
    sns.lineplot(data=df, x='step', y='memory_match', hue='model')
    plt.title('Memory Match Rate by Step')
    plt.xlabel('Step')
    plt.ylabel('Match (1=true, 0=false)')
    plt.show()

if 'memory_distance' in df:
    plt.figure(figsize=(10,4))
    sns.lineplot(data=df, x='step', y='memory_distance', hue='model')
    plt.title('Memory Edit Distance by Step')
    plt.xlabel('Step')
    plt.ylabel('Edit Distance (lower is better)')
    plt.show()
